# Example 1.2.2 / 1: Defining a Parameter with =

Companion notebook for **1.2.2 Computed Parameters** in the AMPL lab Documentation, section "Defining a Parameter with `=`". It runs the page's example from top to bottom: the model, the amplpy way (data.py, run.py), then the pure-AMPL way (data.dat, model.run). Solver: HiGHS.

## Install Dependencies
💡 TIP: Visit [portal.ampl.com](https://portal.ampl.com/user/ampl/license/list) or [dev.ampl.com](https://dev.ampl.com/ampl/install.html) for AMPL setup on Colab, Docker, Python, Windows, Linux, macOS

In [ ]:
# Install dependencies
%pip install -q amplpy

# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["highs"],      # modules to install
    license_uuid="default") # license to use

## Defining a Parameter with `=`

Add `= expression` to the declaration. The expression may use sets and parameters declared
earlier and the declaration's own index, but not variables. Python gives only the data, `rate`;
`to_pandas()` returns the computed values like any other parameter's:

### model.mod
The model, written to a file that both ways of running it read.

In [2]:
%%writefile model.mod
set PROD;                                     # products
param rate {PROD} > 0;                        # tons per hour (data)
param hours_per_ton {p in PROD} = 1 / rate[p]; # computed

Writing model.mod


### amplpy: data.py and run.py

`ampl` is the `AMPL` object created above. Each run starts from a fresh session that has read `model.mod`; then `data.py` sets the data from Python and `run.py` solves and reads the results.

In [3]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.eval("reset options;")  # reset keeps the options
ampl.read("model.mod")

In [4]:
# data.py
ampl.set["PROD"] = ["bands", "coils"]
ampl.param["rate"] = {"bands": 200, "coils": 140}

In [5]:
# run.py
print(ampl.param["hours_per_ton"].to_pandas())

       hours_per_ton
bands       0.005000
coils       0.007143


### Pure AMPL: data.dat and model.run

Pure AMPL:

In [6]:
%%writefile data.dat
set PROD := bands coils;
param rate := bands 200  coils 140;

Writing data.dat


In [7]:
%%ampl_eval
reset;
reset options;
model model.mod;
data data.dat;

In [8]:
%%ampl_eval
# model.run
display hours_per_ton;

hours_per_ton [*] :=
bands  0.005
coils  0.00714286
;

